<a href="https://colab.research.google.com/github/elnazshokrollahi/My-Agentic-Journey/blob/main/RAG_Evaluation/RAG_Evaluation.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
from google.colab import drive
drive.mount('/content/drive')


Mounted at /content/drive


In [2]:
import os
from google.colab import userdata
os.environ["OPENAI_API_KEY"] = userdata.get('OPENAI_APIKEY')
print("Key loaded ✅")

Key loaded ✅


In [4]:
!pip install -q langchain-openai langchain-community langchain-core faiss-cpu openai python-dotenv

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 120.4/120.4 kB 8.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.4/2.4 MB 71.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 18.5/18.5 MB 67.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.0/1.0 MB 49.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 61.7/61.7 kB 4.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 73.1/73.1 kB 6.1 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
google-colab 1.0.0 requires requests==2.32.4, but you have requests 2.34.2 which is incompatible.


In [7]:
# -*- coding: utf-8 -*-
"""
RAG Evaluation Demo
===================

This demo teaches:
1. Two-layer evaluation approach for RAG systems
2. RETRIEVAL LAYER: Precision, Recall, F1 Score
3. GENERATION LAYER: Groundedness (Faithfulness), Response Completeness
4. Using LLM-as-judge for generation metrics
5. Creating comprehensive evaluation reports
"""

import json
import os
import numpy as np
from openai import OpenAI
from langchain_openai import OpenAIEmbeddings, ChatOpenAI
from langchain_community.vectorstores import FAISS
from langchain_core.documents import Document
from dotenv import load_dotenv

# Load environment variables
load_dotenv()

print("="*80)
print("RAG EVALUATION: TWO-LAYER APPROACH")
print("="*80)
print("\nLayer 1: RETRIEVAL (Precision, Recall, F1)")
print("Layer 2: GENERATION (Groundedness, Completeness)")

# ============================================================================
# PART 1: Setup - Load Data and Build RAG System
# ============================================================================
print("\n" + "="*80)
print("PART 1: Setup RAG System for Evaluation")
print("="*80)

# Load tickets
with open('/content/drive/MyDrive/synthetic_tickets.json', 'r', encoding='utf-8') as f:
    tickets = json.load(f)
print(f"✓ Loaded {len(tickets)} support tickets")

# Load evaluation queries with ground truth
with open('/content/drive/MyDrive/evaluation_queries.json', 'r', encoding='utf-8') as f:
    eval_queries = json.load(f)
print(f"✓ Loaded {len(eval_queries)} evaluation queries with ground truth")

# Build documents
documents = []
for ticket in tickets:
    content = f"""Ticket ID: {ticket['ticket_id']}
Title: {ticket['title']}
Category: {ticket['category']}
Priority: {ticket['priority']}
Description: {ticket['description']}
Resolution: {ticket['resolution']}"""

    doc = Document(
        page_content=content,
        metadata={
            'ticket_id': ticket['ticket_id'],
            'title': ticket['title'],
            'category': ticket['category']
        }
    )
    documents.append(doc)

# Initialize embeddings and LLM with timeout
embeddings = OpenAIEmbeddings(
    model=os.getenv('OPENAI_EMBEDDING_MODEL', 'text-embedding-3-small'),
    request_timeout=30
)

# Use direct OpenAI client instead of LangChain ChatOpenAI (fixes connection pooling issue)
openai_client = OpenAI(api_key=os.getenv('OPENAI_API_KEY'), timeout=30.0, max_retries=2)
chat_model = os.getenv('OPENAI_CHAT_MODEL', 'gpt-4o-mini')

print("✓ OpenAI models initialized (30s timeout)")

# Test API connection
print("\nTesting OpenAI API connection...")
try:
    test_response = openai_client.chat.completions.create(
        model=chat_model,
        messages=[{"role": "user", "content": "Say 'OK'"}],
        max_tokens=5
    )
    print("✓ API connection successful")
except Exception as e:
    print(f"✗ API connection failed: {str(e)[:100]}")
    print("\nPlease check:")
    print("  1. OPENAI_API_KEY is set correctly in .env")
    print("  2. You have internet connectivity")
    print("  3. Your API key is valid and has credits")
    exit(1)

# Build vector store
vector_store = FAISS.from_documents(documents, embeddings)
print("✓ Vector store built with FAISS")

# Create simple RAG function
def generate_answer(query, k=3):
    """Retrieve context and generate answer"""
    # Retrieve documents
    docs = vector_store.similarity_search(query, k=k)

    # Build context
    context = "\n\n".join([doc.page_content for doc in docs])

    # Create prompt
    prompt = f"""You are a technical support assistant. Answer the question using ONLY the provided context.

Context:
{context}

Question: {query}

Answer (cite ticket IDs):"""

    # Generate answer using direct OpenAI client
    response = openai_client.chat.completions.create(
        model=chat_model,
        messages=[{"role": "user", "content": prompt}],
        temperature=0
    )
    answer = response.choices[0].message.content

    return {'answer': answer, 'source_documents': docs}

print("✓ RAG function ready")

# ============================================================================
# PART 2: RETRIEVAL LAYER EVALUATION
# ============================================================================
print("\n" + "="*80)
print("PART 2: RETRIEVAL LAYER EVALUATION")
print("="*80)
print("\nMetrics: Precision, Recall, F1 Score")
print("Measures: Are we retrieving the right documents?")

def calculate_retrieval_metrics(retrieved_ids, relevant_ids, k=3):
    """
    Calculate Precision, Recall, and F1 at k

    Precision@k = (relevant docs in top-k) / k
    Recall@k = (relevant docs in top-k) / (total relevant docs)
    F1@k = harmonic mean of Precision and Recall
    """
    retrieved_set = set(retrieved_ids[:k])
    relevant_set = set(relevant_ids)

    # True positives: relevant docs that were retrieved
    tp = len(retrieved_set & relevant_set)

    # Precision: what % of retrieved docs are relevant?
    precision = tp / k if k > 0 else 0

    # Recall: what % of relevant docs were retrieved?
    recall = tp / len(relevant_set) if len(relevant_set) > 0 else 0

    # F1: harmonic mean balancing precision and recall
    f1 = 2 * (precision * recall) / (precision + recall) if (precision + recall) > 0 else 0

    return {
        'precision': precision,
        'recall': recall,
        'f1': f1,
        'true_positives': tp,
        'retrieved_count': k,
        'relevant_count': len(relevant_set)
    }

# Evaluate retrieval for all queries
print("\nEvaluating retrieval for all queries...")
retrieval_results = []

for idx, eval_query in enumerate(eval_queries, 1):
    query = eval_query['question']
    relevant_ids = eval_query['relevant_ticket_ids']

    print(f"  [{idx}/{len(eval_queries)}] Processing query...", end='\r')

    try:
        # Retrieve documents
        results = vector_store.similarity_search(query, k=5)
        retrieved_ids = [doc.metadata['ticket_id'] for doc in results]

        # Calculate metrics at k=3
        metrics = calculate_retrieval_metrics(retrieved_ids, relevant_ids, k=3)
        metrics['query_id'] = eval_query['query_id']
        metrics['question'] = query
        metrics['retrieved'] = retrieved_ids[:3]
        metrics['relevant'] = relevant_ids

        retrieval_results.append(metrics)
    except Exception as e:
        print(f"\n  ⚠ Error on query {idx}: {str(e)[:50]}")
        continue

print(f"\n✓ Completed retrieval evaluation for {len(retrieval_results)} queries")

# Aggregate results
print("\n" + "-"*80)
print("RETRIEVAL METRICS (Averaged across all queries)")
print("-"*80)

avg_precision = np.mean([r['precision'] for r in retrieval_results])
avg_recall = np.mean([r['recall'] for r in retrieval_results])
avg_f1 = np.mean([r['f1'] for r in retrieval_results])

print(f"\nPrecision@3: {avg_precision:.4f}")
print(f"  → What % of retrieved documents are actually relevant?")
print(f"  → Target: > 0.80 for production")

print(f"\nRecall@3:    {avg_recall:.4f}")
print(f"  → What % of all relevant documents did we find?")
print(f"  → Target: > 0.70 for production")

print(f"\nF1 Score@3:  {avg_f1:.4f}")
print(f"  → Balanced measure of retrieval quality")
print(f"  → Target: > 0.75 for production")

# Show per-query breakdown
print("\n" + "-"*80)
print("PER-QUERY RETRIEVAL RESULTS (Sample)")
print("-"*80)

for i in range(min(3, len(retrieval_results))):
    result = retrieval_results[i]
    print(f"\n{result['query_id']}: {result['question'][:60]}...")
    print(f"  Relevant:  {result['relevant']}")
    print(f"  Retrieved: {result['retrieved']}")
    print(f"  Precision: {result['precision']:.2f} | Recall: {result['recall']:.2f} | F1: {result['f1']:.2f}")

# ============================================================================
# PART 3: GENERATION LAYER EVALUATION
# ============================================================================
print("\n" + "="*80)
print("PART 3: GENERATION LAYER EVALUATION")
print("="*80)
print("\nMetrics: Groundedness, Response Completeness")
print("Measures: Is the generated answer faithful and complete?")

# Initialize OpenAI client for LLM-as-judge (reuse the same client)
client = openai_client

def evaluate_groundedness(answer, context_docs):
    """
    Groundedness (Faithfulness): Is the answer supported by the retrieved context?
    Uses LLM-as-judge to check if answer contains hallucinations

    Returns: score 0.0-1.0 (higher = more grounded)
    """
    context = "\n\n".join([doc.page_content for doc in context_docs])

    prompt = f"""Evaluate if the ANSWER is fully supported by the CONTEXT. Check for hallucinations or unsupported claims.

CONTEXT:
{context}

ANSWER:
{answer}

Rate the groundedness from 0 to 10:
- 10: Every claim in the answer is directly supported by the context
- 7-9: Most claims supported, minor unsupported details
- 4-6: Some claims supported, some speculation or external knowledge
- 1-3: Little support from context, mostly hallucinated
- 0: Completely hallucinated, no relation to context

Provide:
1. Score (0-10)
2. Reasoning (one sentence)

Format:
Score: X
Reasoning: <explanation>"""

    try:
        response = client.chat.completions.create(
            model=os.getenv('OPENAI_CHAT_MODEL', 'gpt-4o-mini'),
            messages=[{"role": "user", "content": prompt}],
            temperature=0,
            timeout=30
        )

        output = response.choices[0].message.content

        # Parse score
        try:
            score_line = [line for line in output.split('\n') if line.startswith('Score:')][0]
            score = float(score_line.split(':')[1].strip()) / 10.0  # Normalize to 0-1
        except:
            score = 0.5  # Default if parsing fails

        return {
            'score': score,
            'verdict': 'GROUNDED' if score >= 0.7 else 'PARTIAL' if score >= 0.4 else 'HALLUCINATED',
            'explanation': output
        }
    except Exception as e:
        print(f"\n    ⚠ Error evaluating groundedness: {str(e)[:50]}")
        return {'score': 0.5, 'verdict': 'ERROR', 'explanation': str(e)}

def evaluate_completeness(question, answer, reference_answer=None):
    """
    Response Completeness: Does the answer fully address the question?
    Uses LLM-as-judge to check if all aspects of the question are answered

    Returns: score 0.0-1.0 (higher = more complete)
    """
    if reference_answer:
        prompt = f"""Evaluate if the ANSWER fully addresses the QUESTION compared to the REFERENCE ANSWER.

QUESTION:
{question}

REFERENCE ANSWER:
{reference_answer}

GENERATED ANSWER:
{answer}

Rate the completeness from 0 to 10:
- 10: Fully answers all aspects of the question (as good as reference)
- 7-9: Answers most parts, minor gaps
- 4-6: Partial answer, missing key information
- 1-3: Minimal answer, major gaps
- 0: Does not answer the question

Provide:
1. Score (0-10)
2. Reasoning (one sentence)

Format:
Score: X
Reasoning: <explanation>"""
    else:
        prompt = f"""Evaluate if the ANSWER fully addresses the QUESTION.

QUESTION:
{question}

ANSWER:
{answer}

Rate the completeness from 0 to 10:
- 10: Fully answers all aspects of the question
- 7-9: Answers most parts, minor gaps
- 4-6: Partial answer, missing key information
- 1-3: Minimal answer, major gaps
- 0: Does not answer the question

Provide:
1. Score (0-10)
2. Reasoning (one sentence)

Format:
Score: X
Reasoning: <explanation>"""

    try:
        response = client.chat.completions.create(
            model=os.getenv('OPENAI_CHAT_MODEL', 'gpt-4o-mini'),
            messages=[{"role": "user", "content": prompt}],
            temperature=0,
            timeout=30
        )

        output = response.choices[0].message.content

        # Parse score
        try:
            score_line = [line for line in output.split('\n') if line.startswith('Score:')][0]
            score = float(score_line.split(':')[1].strip()) / 10.0  # Normalize to 0-1
        except:
            score = 0.5  # Default if parsing fails

        return {
            'score': score,
            'verdict': 'COMPLETE' if score >= 0.7 else 'PARTIAL' if score >= 0.4 else 'INCOMPLETE',
            'explanation': output
        }
    except Exception as e:
        print(f"\n    ⚠ Error evaluating completeness: {str(e)[:50]}")
        return {'score': 0.5, 'verdict': 'ERROR', 'explanation': str(e)}

# Evaluate generation for sample queries
print("\nEvaluating generation quality (sample queries)...")
print("Note: This uses LLM-as-judge (GPT-4o-mini) to evaluate quality")
print("This may take 30-60 seconds per query...\n")

generation_results = []

for idx, eval_query in enumerate(eval_queries[:2], 1):  # Evaluate first 2 to save time
    query = eval_query['question']
    reference = eval_query.get('reference_answer', None)

    print("-"*80)
    print(f"\n[{idx}/2] Query: {query}")

    try:
        # Generate answer using RAG
        print("  → Generating answer...")
        result = generate_answer(query)
        answer = result['answer']
        source_docs = result['source_documents']

        print(f"\nGenerated Answer:\n{answer}")

        # Evaluate groundedness
        print("\n  → Evaluating Groundedness...")
        groundedness = evaluate_groundedness(answer, source_docs)
        print(f"    Score: {groundedness['score']:.2f} - {groundedness['verdict']}")

        # Evaluate completeness
        print("\n  → Evaluating Completeness...")
        completeness = evaluate_completeness(query, answer, reference)
        print(f"    Score: {completeness['score']:.2f} - {completeness['verdict']}")

        generation_results.append({
            'query_id': eval_query['query_id'],
            'question': query,
            'answer': answer,
            'groundedness_score': groundedness['score'],
            'completeness_score': completeness['score']
        })
    except Exception as e:
        print(f"\n  ✗ Error processing query: {str(e)[:80]}")
        continue

print(f"\n✓ Completed generation evaluation for {len(generation_results)} queries")

# Aggregate generation metrics
print("\n" + "="*80)
print("GENERATION METRICS (Averaged)")
print("="*80)

avg_groundedness = np.mean([r['groundedness_score'] for r in generation_results])
avg_completeness = np.mean([r['completeness_score'] for r in generation_results])

print(f"\nGroundedness:  {avg_groundedness:.4f}")
print(f"  → Are answers supported by retrieved context?")
print(f"  → Target: > 0.80 (minimize hallucinations)")

print(f"\nCompleteness:  {avg_completeness:.4f}")
print(f"  → Do answers fully address the questions?")
print(f"  → Target: > 0.75 (comprehensive responses)")

# ============================================================================
# PART 4: Comprehensive Evaluation Report
# ============================================================================
print("\n" + "="*80)
print("PART 4: COMPREHENSIVE EVALUATION REPORT")
print("="*80)

print(f"""
┌─────────────────────────────────────────────────────────────┐
│                    RAG SYSTEM EVALUATION                    │
├─────────────────────────────────────────────────────────────┤
│ Dataset: {len(eval_queries)} evaluation queries                           │
├─────────────────────────────────────────────────────────────┤
│ RETRIEVAL LAYER                                             │
│   • Precision@3:  {avg_precision:.4f}  {'✓' if avg_precision >= 0.80 else '⚠' if avg_precision >= 0.70 else '✗'}                              │
│   • Recall@3:     {avg_recall:.4f}  {'✓' if avg_recall >= 0.70 else '⚠' if avg_recall >= 0.60 else '✗'}                              │
│   • F1 Score@3:   {avg_f1:.4f}  {'✓' if avg_f1 >= 0.75 else '⚠' if avg_f1 >= 0.65 else '✗'}                              │
├─────────────────────────────────────────────────────────────┤
│ GENERATION LAYER                                            │
│   • Groundedness: {avg_groundedness:.4f}  {'✓' if avg_groundedness >= 0.80 else '⚠' if avg_groundedness >= 0.70 else '✗'}                              │
│   • Completeness: {avg_completeness:.4f}  {'✓' if avg_completeness >= 0.75 else '⚠' if avg_completeness >= 0.65 else '✗'}                              │
└─────────────────────────────────────────────────────────────┘

INTERPRETATION:
""")

# Provide diagnostic feedback
if avg_precision < 0.70:
    print("⚠ Low Precision → Too much noise in retrieval")
    print("  Fix: Improve chunking, use metadata filtering, or increase similarity threshold")

if avg_recall < 0.60:
    print("⚠ Low Recall → Missing relevant documents")
    print("  Fix: Increase k, improve embedding model, or use hybrid retrieval")

if avg_f1 < 0.65:
    print("⚠ Low F1 → Overall retrieval quality needs improvement")
    print("  Fix: Balance precision and recall by tuning retrieval parameters")

if avg_groundedness < 0.70:
    print("⚠ Low Groundedness → System is hallucinating")
    print("  Fix: Strengthen prompt instructions, use stricter grounding, or improve context quality")

if avg_completeness < 0.65:
    print("⚠ Low Completeness → Answers are incomplete")
    print("  Fix: Retrieve more context (increase k), improve prompt, or use better LLM")

if all([avg_precision >= 0.80, avg_recall >= 0.70, avg_f1 >= 0.75,
        avg_groundedness >= 0.80, avg_completeness >= 0.75]):
    print("✓ All metrics in target range - System is production-ready!")

# ============================================================================
# PART 5: Comparing Different RAG Configurations
# ============================================================================
print("\n" + "="*80)
print("PART 5: A/B Testing Framework")
print("="*80)

print("\nExample: Compare retrieval with k=3 vs k=5")

def compare_configurations(queries, k_values):
    """Compare different retrieval configurations"""
    results = {}

    for k in k_values:
        print(f"  Testing k={k}...")
        retrievals = []
        for query in queries[:3]:  # Test subset
            try:
                docs = vector_store.similarity_search(query['question'], k=k)
                retrieved_ids = [doc.metadata['ticket_id'] for doc in docs]
                metrics = calculate_retrieval_metrics(retrieved_ids, query['relevant_ticket_ids'], k=k)
                retrievals.append(metrics)
            except Exception as e:
                print(f"\n    ⚠ Error: {str(e)[:50]}")
                continue

        if retrievals:
            results[f'k={k}'] = {
                'precision': np.mean([r['precision'] for r in retrievals]),
                'recall': np.mean([r['recall'] for r in retrievals]),
                'f1': np.mean([r['f1'] for r in retrievals])
            }

    return results

comparison = compare_configurations(eval_queries, [3, 5])

print("\nConfiguration Comparison:")
print(f"{'Config':<10} {'Precision':<12} {'Recall':<12} {'F1':<12}")
print("-" * 46)
for config, metrics in comparison.items():
    print(f"{config:<10} {metrics['precision']:<12.4f} {metrics['recall']:<12.4f} {metrics['f1']:<12.4f}")

print("\n" + "="*80)
print("DEMO COMPLETE!")
print("="*80)

print("""
Key Takeaways:
──────────────
1. Two-Layer Evaluation is Essential
   → Separately measure retrieval and generation quality

2. Retrieval Metrics (Precision, Recall, F1)
   → Diagnose if you're finding the right documents

3. Generation Metrics (Groundedness, Completeness)
   → Ensure faithful and comprehensive answers

4. Use LLM-as-Judge for Generation Metrics
   → Automated evaluation using GPT-4/Claude

5. Always Create Evaluation Datasets
   → Ground truth enables systematic improvement

6. A/B Test Different Configurations
   → Measure impact of changes quantitatively

Production Targets:
───────────────────
✓ Precision@3 > 0.80
✓ Recall@3 > 0.70
✓ F1@3 > 0.75
✓ Groundedness > 0.80
✓ Completeness > 0.75
""")

RAG EVALUATION: TWO-LAYER APPROACH

Layer 1: RETRIEVAL (Precision, Recall, F1)
Layer 2: GENERATION (Groundedness, Completeness)

PART 1: Setup RAG System for Evaluation
✓ Loaded 20 support tickets
✓ Loaded 15 evaluation queries with ground truth
✓ OpenAI models initialized (30s timeout)

Testing OpenAI API connection...
✓ API connection successful
✓ Vector store built with FAISS
✓ RAG function ready

PART 2: RETRIEVAL LAYER EVALUATION

Metrics: Precision, Recall, F1 Score
Measures: Are we retrieving the right documents?

Evaluating retrieval for all queries...
  [15/15] Processing query...
✓ Completed retrieval evaluation for 15 queries

--------------------------------------------------------------------------------
RETRIEVAL METRICS (Averaged across all queries)
--------------------------------------------------------------------------------

Precision@3: 0.4444
  → What % of retrieved documents are actually relevant?
  → Target: > 0.80 for production

Recall@3:    0.9444
  → What % 

In [10]:
from collections import defaultdict

buckets = defaultdict(set)
for d in vector_store.similarity_search("system", k=50):
    buckets[d.metadata['category']].add(d.metadata['ticket_id'])

for cat, ids in buckets.items():
    print(f"{cat}: {sorted(ids)}")

Authentication: ['TICK-001', 'TICK-011', 'TICK-014', 'TICK-016']
Performance: ['TICK-005', 'TICK-010']
Database: ['TICK-002']
Real-time: ['TICK-017']
User Management: ['TICK-020']
Mobile: ['TICK-004']
Reporting: ['TICK-012']
Search: ['TICK-007']
API: ['TICK-009', 'TICK-018']
Integration: ['TICK-013']
Payment: ['TICK-003']
Email: ['TICK-006']
Export: ['TICK-015']
Media Processing: ['TICK-019']
File Upload: ['TICK-008']


In [11]:
eval_queries = [
    {"question": "How do I fix authentication failures?", "relevant_ticket_ids": sorted(buckets["Authentication"])},
    {"question": "Why is the dashboard slow?",            "relevant_ticket_ids": sorted(buckets["Performance"])},
    {"question": "Payment processing problems",           "relevant_ticket_ids": sorted(buckets["Payment"])},
]

for q in eval_queries:
    print(q["question"], "→", q["relevant_ticket_ids"])

How do I fix authentication failures? → ['TICK-001', 'TICK-011', 'TICK-014', 'TICK-016']
Why is the dashboard slow? → ['TICK-005', 'TICK-010']
Payment processing problems → ['TICK-003']


In [13]:
def calculate_metrics(retrieved_ids, relevant_ids, k=3):
    """Precision@k, Recall@k, F1@k"""
    retrieved_set = set(retrieved_ids[:k])
    relevant_set = set(relevant_ids)

    tp = len(retrieved_set & relevant_set)

    precision = tp / k if k > 0 else 0
    recall = tp / len(relevant_set) if len(relevant_set) > 0 else 0
    f1 = 2 * (precision * recall) / (precision + recall) if (precision + recall) > 0 else 0

    return {'precision': precision, 'recall': recall, 'f1': f1}

In [14]:
all_metrics = []
for query in eval_queries:
    docs = vector_store.similarity_search(query['question'], k=3)
    retrieved = [doc.metadata['ticket_id'] for doc in docs]
    metrics = calculate_metrics(retrieved, query['relevant_ticket_ids'])
    all_metrics.append(metrics)

avg_precision = np.mean([m['precision'] for m in all_metrics])
avg_recall    = np.mean([m['recall'] for m in all_metrics])
avg_f1        = np.mean([m['f1'] for m in all_metrics])

print(f"Precision@3: {avg_precision:.4f}")
print(f"Recall@3:    {avg_recall:.4f}")
print(f"F1@3:        {avg_f1:.4f}")

Precision@3: 0.5556
Recall@3:    0.7500
F1@3:        0.5857


In [18]:
for k in [1, 3, 5, 10]:
    metrics = []
    for query in eval_queries:
        docs = vector_store.similarity_search(query['question'], k=k)
        retrieved = [doc.metadata['ticket_id'] for doc in docs]
        m = calculate_metrics(retrieved, query['relevant_ticket_ids'], k=k)
        metrics.append(m)

    print(f"k={k}: Precision={np.mean([m['precision'] for m in metrics]):.2f}, "
          f"Recall={np.mean([m['recall'] for m in metrics]):.2f}, "
          f"F1={np.mean([m['f1'] for m in metrics]):.2f}")

k=1: Precision=1.00, Recall=0.58, F1=0.69
k=3: Precision=0.56, Recall=0.75, F1=0.59
k=5: Precision=0.40, Recall=0.83, F1=0.50
k=10: Precision=0.23, Recall=1.00, F1=0.36


In [19]:
def average_precision(retrieved_ids, relevant_ids):
    """
    AP = average of precision@k for all k where a relevant doc is found
    Higher score = relevant docs appear earlier in results
    """
    precisions = []
    relevant_count = 0

    for k, doc_id in enumerate(retrieved_ids, 1):
        if doc_id in relevant_ids:
            relevant_count += 1
            precision_at_k = relevant_count / k
            precisions.append(precision_at_k)

    return np.mean(precisions) if precisions else 0.0

In [21]:
retrieved=['A', 'B', 'C']
relevant=['A', 'C']

In [22]:
average_precision('retrieved', 'relevant')

np.float64(0.9379251700680271)

In [25]:
# --- Test cases (predict each before reading the output!) ---
print("Test 1 [A,B,C], rel=[A,C]:", round(average_precision(['A','B','C'], ['A','C']), 2))  # expect 0.83
print("Test 2 [B,A,C], rel=[A,C]:", round(average_precision(['B','A','C'], ['A','C']), 2))  # expect 0.58
print("Test 3 [A,C,B], rel=[A,C]:", round(average_precision(['A','C','B'], ['A','C']), 2))  # expect 1.00
print("Test 4 [B,X,Y], rel=[A,C]:", round(average_precision(['B','X','Y'], ['A','C']), 2))  # expect 0.00


Test 1 [A,B,C], rel=[A,C]: 0.83
Test 2 [B,A,C], rel=[A,C]: 0.58
Test 3 [A,C,B], rel=[A,C]: 1.0
Test 4 [B,X,Y], rel=[A,C]: 0.0


In [26]:
def evaluate_groundedness(answer, context_docs):
    """Ask an LLM: is this answer supported by the context?"""
    context = "\n\n".join([doc.page_content for doc in context_docs])
    prompt = f"""Evaluate if the ANSWER is fully supported by the CONTEXT.

CONTEXT:
{context}

ANSWER:
{answer}

Rate groundedness (0-10):
- 10: Every claim is supported
- 5: Some claims unsupported
- 0: Completely hallucinated

Format: Score: X / Reason: <explanation>"""
    response = openai_client.chat.completions.create(
        model='gpt-4o-mini',
        messages=[{"role": "user", "content": prompt}],
        temperature=0
    )
    return response.choices[0].message.content


# --- Get a REAL answer from your RAG ---
result = generate_answer("How do I fix authentication failures?")
print("RAG ANSWER:\n", result['answer'])

print("\n--- JUDGE ON THE REAL ANSWER (should score HIGH) ---")
print(evaluate_groundedness(result['answer'], result['source_documents']))

# --- Now feed the SAME context a HALLUCINATED answer (should score LOW) ---
fake_answer = ("To fix authentication, restart the quantum encryption module and "
               "call support at 1-800-555-0199. Also, all passwords must contain an emoji.")
print("\n--- JUDGE ON A FAKE ANSWER (should score LOW) ---")
print(evaluate_groundedness(fake_answer, result['source_documents']))

RAG ANSWER:
 To fix authentication failures, you can refer to the following resolutions from the tickets:

1. For users unable to log in after a password reset (Ticket ID: TICK-001), clear all active sessions and force re-authentication. Implement automatic session cleanup on password change due to the updated password hash algorithm.

2. For issues with two-factor authentication codes not working (Ticket ID: TICK-016), ensure that the server time is correctly synchronized. Configure NTP synchronization and verify time accuracy, as TOTP codes are time-sensitive.

3. For Single Sign-On (SSO) authentication issues (Ticket ID: TICK-011), update the IdP configuration to match the new default SAML signature algorithm (SHA-256) after the v2.5 upgrade. 

By addressing these specific issues, you can resolve authentication failures.

--- JUDGE ON THE REAL ANSWER (should score HIGH) ---
Score: 10 / Reason: Every claim in the ANSWER is fully supported by the CONTEXT. Each resolution provided corr

In [27]:
def evaluate_completeness(question, answer, reference_answer=None):
    """Ask an LLM: does this answer fully address the question?"""
    prompt = f"""Evaluate if the ANSWER fully addresses the QUESTION.

QUESTION: {question}
ANSWER: {answer}

Rate completeness (0-10):
- 10: Fully answers all aspects
- 5: Partial answer
- 0: Does not answer

Format: Score: X / Reason: <explanation>"""
    response = openai_client.chat.completions.create(
        model='gpt-4o-mini',
        messages=[{"role": "user", "content": prompt}],
        temperature=0
    )
    return response.choices[0].message.content


question = "How do I fix authentication failures?"

# Full RAG answer
result = generate_answer(question)
print("--- JUDGE ON THE FULL RAG ANSWER (should score HIGH) ---")
print(evaluate_completeness(question, result['answer']))

# Deliberately thin answer (should score LOW)
thin_answer = "Try resetting your password."
print("\n--- JUDGE ON A THIN ANSWER (should score LOW) ---")
print(evaluate_completeness(question, thin_answer))

--- JUDGE ON THE FULL RAG ANSWER (should score HIGH) ---
Score: 10 / Reason: The ANSWER provides specific and actionable resolutions for various types of authentication failures, addressing different scenarios such as password resets, two-factor authentication issues, and Single Sign-On problems. Each resolution is clearly linked to a ticket ID, which adds context and credibility. Overall, it comprehensively covers the question of how to fix authentication failures.

--- JUDGE ON A THIN ANSWER (should score LOW) ---
Score: 5 / Reason: The answer provides a potential solution (resetting the password) for authentication failures, but it does not address other possible causes or solutions, such as checking username, ensuring the account is not locked, or verifying network issues. Therefore, it is a partial answer.


In [28]:
from collections import defaultdict

# add a category tag to each query so failures can be grouped
eval_queries = [
    {"question": "How do I fix authentication failures?", "relevant_ticket_ids": sorted(buckets["Authentication"]), "category": "Authentication"},
    {"question": "Why is the dashboard slow?",            "relevant_ticket_ids": sorted(buckets["Performance"]),    "category": "Performance"},
    {"question": "Payment processing problems",           "relevant_ticket_ids": sorted(buckets["Payment"]),        "category": "Payment"},
]

def analyze_failures(eval_queries, vector_store, threshold=0.5):
    failures = []
    for query in eval_queries:
        docs = vector_store.similarity_search(query['question'], k=3)
        retrieved = [doc.metadata['ticket_id'] for doc in docs]
        metrics = calculate_metrics(retrieved, query['relevant_ticket_ids'])
        if metrics['precision'] < threshold:
            failures.append({
                'query': query['question'],
                'precision': metrics['precision'],
                'retrieved': retrieved,
                'expected': query['relevant_ticket_ids'],
                'category': query.get('category', 'Unknown')
            })

    by_category = defaultdict(list)
    for f in failures:
        by_category[f['category']].append(f)

    print(f"Found {len(failures)} failures (precision < {threshold})\n")
    for f in failures:
        print(f"  '{f['query']}'  (precision {f['precision']:.2f})")
        print(f"     retrieved: {f['retrieved']}")
        print(f"     expected:  {f['expected']}\n")
    for category, items in by_category.items():
        print(f"  {category}: {len(items)} failures")
    return failures


failures = analyze_failures(eval_queries, vector_store)

Found 2 failures (precision < 0.5)

  'Why is the dashboard slow?'  (precision 0.33)
     retrieved: ['TICK-010', 'TICK-017', 'TICK-018']
     expected:  ['TICK-005', 'TICK-010']

  'Payment processing problems'  (precision 0.33)
     retrieved: ['TICK-003', 'TICK-008', 'TICK-009']
     expected:  ['TICK-003']

  Performance: 1 failures
  Payment: 1 failures


In [29]:
import time

class RAGMetrics:
    def __init__(self):
        self.query_times = []
        self.token_counts = []

    def track_query(self, query, result, elapsed_time):
        self.query_times.append(elapsed_time)
        tokens = len(result) / 4          # rough: ~4 chars per token (output only)
        self.token_counts.append(tokens)

    def report(self):
        print(f"Total queries: {len(self.query_times)}")
        print(f"Avg latency:  {np.mean(self.query_times):.3f}s")
        print(f"p95 latency:  {np.percentile(self.query_times, 95):.3f}s")
        total_tokens = sum(self.token_counts)
        cost = (total_tokens / 1000) * 0.002
        print(f"Est. cost:    ${cost:.4f}")

metrics = RAGMetrics()
for query in eval_queries[:10]:
    start = time.time()
    result = generate_answer(query['question'])
    elapsed = time.time() - start
    metrics.track_query(query['question'], result['answer'], elapsed)

metrics.report()

Total queries: 3
Avg latency:  2.000s
p95 latency:  3.203s
Est. cost:    $0.0007


In [30]:
def parse_score(judge_text):
    """Pull the 0-10 number out of the judge's 'Score: X / Reason: ...' text, normalize to 0-1."""
    try:
        # find the line/segment starting with 'Score:' and grab the number after it
        after = judge_text.split("Score:")[1]
        number = after.split("/")[0].strip().split()[0]   # first token after 'Score:'
        return float(number) / 10.0
    except Exception:
        return 0.5   # safe fallback if parsing fails


def evaluation_report(eval_queries, vector_store, generate_fn):
    """Generate a comprehensive two-layer evaluation report."""
    retrieval_scores = []
    groundedness_scores = []

    for query in eval_queries[:5]:
        # --- Retrieval layer ---
        docs = vector_store.similarity_search(query['question'], k=3)
        retrieved = [doc.metadata['ticket_id'] for doc in docs]
        r_metrics = calculate_metrics(retrieved, query['relevant_ticket_ids'])
        retrieval_scores.append(r_metrics)

        # --- Generation layer ---
        result = generate_fn(query['question'])
        judge_text = evaluate_groundedness(result['answer'], docs)
        groundedness_scores.append(parse_score(judge_text))   # ← number, not text

    avg_p = np.mean([s['precision'] for s in retrieval_scores])
    avg_r = np.mean([s['recall'] for s in retrieval_scores])
    avg_f1 = np.mean([s['f1'] for s in retrieval_scores])
    avg_g = np.mean(groundedness_scores)

    print("=" * 60)
    print("RAG SYSTEM EVALUATION REPORT")
    print("=" * 60)
    print(f"\nRETRIEVAL LAYER")
    print(f"  Precision@3:  {avg_p:.3f}  {'✓' if avg_p >= 0.80 else '⚠' if avg_p >= 0.70 else '✗'}")
    print(f"  Recall@3:     {avg_r:.3f}  {'✓' if avg_r >= 0.70 else '⚠' if avg_r >= 0.60 else '✗'}")
    print(f"  F1@3:         {avg_f1:.3f}  {'✓' if avg_f1 >= 0.75 else '⚠' if avg_f1 >= 0.65 else '✗'}")
    print(f"\nGENERATION LAYER")
    print(f"  Groundedness: {avg_g:.3f}  {'✓' if avg_g >= 0.80 else '⚠' if avg_g >= 0.70 else '✗'}")
    print("=" * 60)

    if avg_p < 0.70:
        print("⚠ Retrieval precision low → tune k, add filtering, or fix the answer key")
    if avg_g < 0.80:
        print("⚠ Groundedness low → strengthen the prompt's 'use only context' rule")

    return {'precision': avg_p, 'recall': avg_r, 'f1': avg_f1, 'groundedness': avg_g}

report = evaluation_report(eval_queries, vector_store, generate_answer)

RAG SYSTEM EVALUATION REPORT

RETRIEVAL LAYER
  Precision@3:  0.556  ✗
  Recall@3:     0.750  ✓
  F1@3:         0.586  ✗

GENERATION LAYER
  Groundedness: 1.000  ✓
⚠ Retrieval precision low → tune k, add filtering, or fix the answer key


## Key Concepts Summary

| Metric | What It Measures | Target |
|--------|------------------|--------|
| **Precision@k** | % of retrieved docs that are relevant | > 0.80 |
| **Recall@k** | % of relevant docs that were retrieved | > 0.70 |
| **F1@k** | Harmonic mean of precision & recall | > 0.75 |
| **Groundedness** | Is answer supported by context? | > 0.80 |
| **Completeness** | Does answer fully address question? | > 0.75 |

---

## 🎉 Congratulations!

You've completed the RAG Evaluation module! You now know how to:

1. Calculate retrieval metrics (Precision, Recall, F1)
2. Use LLM-as-judge for generation quality
3. Identify and analyze failures
4. Track operational metrics (latency, cost)
5. Build comprehensive evaluation reports